In [1]:
import numpy as np
c = 1 * 10**-12
r1 = 1 # ohms
r2 = 1 # ohms
f = 60 # hertz
w = 2 * np.pi * f
zc = 1 / (w * c * 1j)
z = (r1 * zc) / (r1 + c) + r2
print(z)
print("z = ", abs(z), "@", np.angle(z))

(1-2652582384.86227j)
z =  2652582384.86227 @ -1.5707963264179055


In [2]:
import sympy as sp
c = sp.Symbol('c')
a = np.deg2rad(90)
z = sp.Symbol('z')
zc = sp.Symbol('zc')
eq = [sp.Eq(a, sp.arg(z)),
      sp.Eq(zc, 1 / (w * c * 1j)),
      sp.Eq(z, (r1 * zc) / (r1 + zc) + r2)]
sol = sp.solve(eq, [c, z, zc])
print(sol)

[]


In [3]:
from scipy.optimize import minimize_scalar

r1 = 1 * 10**3
r2 = r2
def z_total(c):
    zc = 1 / (1j * w * c)
    return (r1 * zc) / (r1 + zc) + r2

def phase_deg(c):
    return np.rad2deg(np.angle(z_total(c)))

def error(c):
    return abs(phase_deg(c) - (-90))

result = minimize_scalar(
    error,
    bounds=(1e-12, 1e-3),   # 1 pF to 1000 uF
    method="bounded"
)

c_best = result.x
z_best = z_total(c_best)

print("Best C:", c_best, "F")
print("Best C:", c_best * 1e9, "nF")
print("Phase:", phase_deg(c_best), "degrees")
print("Z:", z_best)
print("Error:", result.fun)

Best C: 8.306156802174464e-05 F
Best C: 83061.56802174464 nF
Phase: -86.37911996400388 degrees
Z: (2.018813804833923-31.902599004861695j)
Error: 3.620880035996123


In [4]:
zc = 1 / (w * c_best * 1j)
r_zc = (r2 * zc) / (r2 + zc)
v = 1.768 # v (rms)
v_r1 = v * r1 / (r1 + r_zc)
print("Voltage across R1:", v_r1, "V (rms)")

Voltage across R1: (1.7662354929371273+5.519764737787275e-05j) V (rms)


In [5]:
vs = 5 # volts
r1 = 1 # ohms
r2 = 1 # ohms
vn = 1 # volts

i_s = 5 # amps

theta_target = np.deg2rad(15)
i2 = (1 / np.cos(theta_target)) * np.exp(1j * theta_target) # amps
print("i2 = ", abs(i2), "@", np.rad2deg(np.angle(i2)), " amps")

n = 1 # volts

def calc_rc(vs, vn, r2, i2):
    invert_rc = (vs - vn) / r2 - i2 / n
    return 1/invert_rc

zc = calc_rc(vs, vn, r2, i2)
print("zc = ", zc, " ohms")

c = 1 / (w * zc * 1j)
print("c = ", c, " F")

i2 =  1.035276180410083 @ 14.999999999999998  amps
zc =  (0.33069523889820446+0.02953650740119702j)  ohms
c =  (-0.0007107573078815774-0.007957747154594769j)  F


In [6]:
def calc_vn_i2(c):
    zc = 1 / (1j * w * c)
    vn = (vs / r1) * (zc * r2) / (r2 + zc)
    return vn, vn / r2

for i in [-12, -9, -6, -3, 0]:
    c = 1 * 10**i # F
    vn, i2 = calc_vn_i2(1 * 10**i)
    print(f"At {c} F, vn = {vn} volts, i2 = {abs(i2)} @ {np.rad2deg(np.angle(i2))} amps")

At 1e-12 F, vn = (5-1.8849555921538754e-09j) volts, i2 = 5.0 @ -2.1599999999999995e-08 amps
At 1e-09 F, vn = (4.9999999999992895-1.8849555921536077e-06j) volts, i2 = 4.999999999999645 @ -2.1599999999998973e-05 amps
At 1e-06 F, vn = (4.999999289388584-0.0018849553242596831j) volts, i2 = 4.99999964469428 @ -0.0215999989767195 amps
At 0.001 F, vn = (4.377814867306127-1.6503973231086122j) volts, i2 = 4.678576101393525 @ -20.655997382339656 amps
At 1 F, vn = (3.518071900413875e-05-0.013262818604569095j) volts, i2 = 0.013262865264364777 @ -89.84801858099271 amps


In [9]:
c = 0.001 # farad
vn, i2 = calc_vn_i2(c)
v_r1 = vn - vs
print(f"v @ r1 = {abs(v_r1)} @ {np.rad2deg(np.angle(v_r1))}")
print(f"v @ r2 = {abs(vn)} @ {np.rad2deg(np.angle(vn))}")

v @ r1 = 1.7637816371278405 @ -110.65599738233965
v @ r2 = 4.678576101393525 @ -20.655997382339656
